In [1]:
import time
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import confusion_matrix
from tqdm import tqdm

/home/bang/projects/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# 1. SETUP PROTECTAI MODEL
# ============================================================
MODEL_NAME = "protectai/deberta-v3-base-prompt-injection-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"[*] Loading ProtectAI model on {device}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

def predict_protectai(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    # Label 1 là 'INJECTION', Label 0 là 'SAFE'
    probabilities = torch.softmax(outputs.logits, dim=1)
    prediction = torch.argmax(probabilities, dim=1).item()
    confidence = probabilities[0][prediction].item()
    return prediction, confidence

# ============================================================
# 2. LOAD DATASET (Same 100 samples as Layer 3 test)
# ============================================================
print("[*] Loading 100 samples from Deepset...")
ds = load_dataset("deepset/prompt-injections", split="train").select(range(0, 100))

# ============================================================
# 3. RUN BENCHMARK
# ============================================================
results = []
print(f"[*] Benchmarking ProtectAI...")

for item in tqdm(ds):
    text = item['text']
    gt = int(item['label'])
    
    t0 = time.perf_counter()
    pred, conf = predict_protectai(text)
    t1 = time.perf_counter()
    latency_ms = (t1 - t0) * 1000.0
    
    results.append({
        "ground_truth": gt,
        "prediction": pred,
        "confidence": conf,
        "latency_ms": latency_ms
    })

# ============================================================
# 4. EVALUATION METRICS
# ============================================================
df = pd.DataFrame(results)
y_true = df['ground_truth']
y_pred = df['prediction']

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
tpr = tp / (tp + fn) if (tp + fn) else 0.0
fpr = fp / (fp + tn) if (fp + tn) else 0.0
precision = tp / (tp + fp) if (tp + fp) else 0.0
accuracy = (tp + tn) / (tp + tn + fp + fn)

print("\n" + "="*60)
print("PROTECTAI DEBERTA-V2 PERFORMANCE METRICS")
print("="*60)
print(f"Confusion Matrix: TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"TPR (Recall): {tpr:.4f}")
print(f"FPR (Fall-out): {fpr:.4f}")
print(f"Precision:    {precision:.4f}")
print(f"Accuracy:     {accuracy:.4f}")
print(f"Mean Latency: {df['latency_ms'].mean():.4f} ms")
print("="*60)

[*] Loading ProtectAI model on cuda...
[*] Loading 100 samples from Deepset...
[*] Benchmarking ProtectAI...


100%|██████████| 100/100 [00:01<00:00, 50.10it/s]


PROTECTAI DEBERTA-V2 PERFORMANCE METRICS
Confusion Matrix: TN=85 FP=0 FN=10 TP=5
TPR (Recall): 0.3333
FPR (Fall-out): 0.0000
Precision:    1.0000
Accuracy:     0.9000
Mean Latency: 19.7884 ms


In [7]:
from layer_3 import run_layer3_eval

run_layer3_eval()

[*] Loading deepset/prompt-injections dataset (train split)...
[*] Warmup 20 calls...
[*] Evaluating 100 samples at Layer 3 using model=gpt-4o-mini ...


100%|██████████| 100/100 [02:19<00:00,  1.39s/it]


LAYER 3 (LLM AUDITOR) PERFORMANCE METRICS
Total Samples: 100
Confusion Matrix: TN=83 FP=2 FN=2 TP=13
TPR (Recall/Sensitivity): 0.8667
FPR (Fall-out):           0.0235
Precision:               0.8667
Accuracy:                0.9600
[OK] Detailed results saved to: layer3_deepset_eval_results.csv
[OK] No API/format errors detected.

Latency (ms):
count     100.000000
mean     1349.705922
std       254.162166
min       976.903867
50%      1320.492690
90%      1581.336731
95%      1767.593041
99%      2293.578874
max      2686.699736


In [6]:
import time
import torch
import pandas as pd
import statistics
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
from tabulate import tabulate

# ============================================================
# 1. CẤU HÌNH HỆ THỐNG & DANH SÁCH MODEL
# ============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Danh sách các mô hình SOTA trên Hugging Face để đối chiếu
MODELS_HF = {
    "ProtectAI-v2": "protectai/deberta-v3-base-prompt-injection-v2",
    # "JasperAI": "JasperLS/deberta-v3-base-injection",
    "Deepset-DeBERTa": "deepset/deberta-v3-base-injection"
}

# ============================================================
# 2. HÀM DỰ ĐOÁN (INFERENCE ENGINE)
# ============================================================
class HFClassifier:
    def __init__(self, model_path):
        print(f"[*] Loading {model_path}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_path).to(DEVICE)
        self.model.eval()

    def detect(self, text):
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad():
            outputs = self.model(**inputs)
        
        # Hầu hết các classifier này: Label 1 = Injection, Label 0 = Safe
        probabilities = torch.softmax(outputs.logits, dim=1)
        prediction = torch.argmax(probabilities, dim=1).item()
        return prediction

# ============================================================
# 3. HÀM TÍNH TOÁN ĐỘ ĐO (METRICS)
# ============================================================
def compute_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "TPR": tp / (tp + fn) if (tp + fn) else 0.0,
        "FPR": fp / (fp + tn) if (fp + tn) else 0.0,
        "ACC": (tp + tn) / (tp + tn + fp + fn)
    }

# ============================================================
# 4. QUY TRÌNH BENCHMARK CHÍNH
# ============================================================
def run_hf_benchmark(samples_n=100):
    # Nạp 100 mẫu từ Deepset
    print(f"[*] Loading {samples_n} samples from deepset/prompt-injections...")
    ds = load_dataset("deepset/prompt-injections", split="train").select(range(samples_n))
    
    final_report = []

    for name, path in MODELS_HF.items():
        detector = HFClassifier(path)
        y_true, y_pred, latencies = [], [], []
        
        print(f"[*] Testing {name}...")
        for item in tqdm(ds):
            text, gt = item['text'], int(item['label'])
            
            # Đo độ trễ L
            t0 = time.perf_counter()
            pred = detector.detect(text)
            t1 = time.perf_counter()
            
            latencies.append((t1 - t0) * 1000) # ms
            y_true.append(gt)
            y_pred.append(pred)
        
        metrics = compute_metrics(y_true, y_pred)
        
        final_report.append({
            "Model": name,
            "TPR (Recall)": f"{metrics['TPR']:.4f}",
            "FPR": f"{metrics['FPR']:.4f}",
            "Accuracy": f"{metrics['ACC']:.4f}",
            "Avg_Latency_ms": f"{statistics.mean(latencies):.2f}",
            "P95_ms": f"{statistics.quantiles(latencies, n=20)[18]:.2f}"
        })

    return pd.DataFrame(final_report)

if __name__ == "__main__":
    report_df = run_hf_benchmark(100)
    
    print("\n" + "="*85)
    print("HUGGING FACE SOTA CLASSIFIERS BENCHMARK (100 DEEPSET SAMPLES)")
    print("="*85)
    print(tabulate(report_df, headers='keys', tablefmt='psql', showindex=False))

[*] Loading 100 samples from deepset/prompt-injections...
[*] Loading protectai/deberta-v3-base-prompt-injection-v2...
[*] Testing ProtectAI-v2...


100%|██████████| 100/100 [00:01<00:00, 90.20it/s]


[*] Loading deepset/deberta-v3-base-injection...
[*] Testing Deepset-DeBERTa...


100%|██████████| 100/100 [00:01<00:00, 88.12it/s]



HUGGING FACE SOTA CLASSIFIERS BENCHMARK (100 DEEPSET SAMPLES)
+-----------------+----------------+-------+------------+------------------+----------+
| Model           |   TPR (Recall) |   FPR |   Accuracy |   Avg_Latency_ms |   P95_ms |
|-----------------+----------------+-------+------------+------------------+----------|
| ProtectAI-v2    |         0.3333 |     0 |        0.9 |            10.96 |    12.27 |
| Deepset-DeBERTa |         1      |     0 |        1   |            11.22 |    14.61 |
+-----------------+----------------+-------+------------+------------------+----------+
